# Pixels to Predictions: CS-GY 6953 (Deep Learning) Kaggle Competition
## MODEL 5
### Author: Mariia Onokhina

---
### Experiment Results
* **Model:** ...
* **Inference:** ...
* **Prompt:** `...
* **Scoring:** ...
* **Prediction:** ...
* **Validation accuracy:** ...
* **Result:** ...

---
**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

## Installation of Libraries 

Before installing any libraries, I create a new Conda environment and add it to Jupyter Notebook to ensure I start from a clean slate and that the code is reproducible.

**Run the following code in a terminal if you'd like to start fresh with a new environment:**
```bash
conda create -n pixels-to-predictions python=3.10
conda activate pixels-to-predictions
conda install -c conda-forge notebook
conda install -c conda-forge ipykernel
python -m ipykernel install --user --name pixels-to-predictions --display-name "Pixels-to-predictions DL"
```

IMPORTANT: Manually change the Kernel in Jupyter Notebook in VS Code or Jupyter Lab to "Pixels-to-predictions DL".

In [1]:
# Uncomment this cell to install the necessary Python packages.
import sys
print("Python:", sys.executable)
!{sys.executable} -m pip install -q transformers==4.57.6 peft==0.18.1 kaggle matplotlib scikit-learn pandas numpy ipywidgets jupyterlab_widgets bitsandbytes accelerate datasets pillow --quiet

Python: /home/devvingduo/miniforge3/envs/pixels-to-predictions/bin/python


---
## Imports & Configuration

In [2]:
# Imports & Configuration
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import ast
from transformers import AutoProcessor, AutoModelForVision2Seq

# For LoRa fine-tuning
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm   # For progress bar
from itertools import combinations

In [3]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [4]:
# This code makes sure it uses only 1 GPU of my choice
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
# Paths 
DATA_DIR = Path("../data")
IMAGE_ROOT = DATA_DIR / "images"

# Model
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Basic Settings
IMG_SIZE = 224

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


---
### Load and Preprocess Data

Download data from https://www.kaggle.com/competitions/pixels-to-predictions/data via Kaggle CLI.

For this, you need a Legacy API key which you can create here: https://www.kaggle.com/settings.

When you create a new key, it will download a ```kaggle.json```. 

In your terminal, run:
```bash
mkdir -p ~/.kaggle
mv <your_downloads_folder> ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json
```

Verify that it worked by running: 
```bash
ls -la ~/.kaggle
```

I have to add it manually because I'm working via SSH into my Linux server machine.

In [7]:
# Uncomment this cell to download the data in a .zip file
#!kaggle competitions download -c pixels-to-predictions

In [8]:
# Uncomment this cell to unzip the data into "data" folder
#!unzip pixels-to-predictions.zip -d data

In [9]:
# Load CSVs
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

Inspect the data. 

In [10]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3109 entries, 0 to 3108
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           3109 non-null   object
 1   image_path   3109 non-null   object
 2   question     3109 non-null   object
 3   choices      3109 non-null   object
 4   num_choices  3109 non-null   int64 
 5   answer       3109 non-null   int64 
 6   hint         2385 non-null   object
 7   lecture      2669 non-null   object
 8   solution     2580 non-null   object
 9   task         3109 non-null   object
 10  grade        3109 non-null   object
 11  subject      3109 non-null   object
 12  topic        3109 non-null   object
 13  category     3109 non-null   object
 14  skill        3109 non-null   object
dtypes: int64(2), object(13)
memory usage: 364.5+ KB


In [11]:
# Function to parse the choices column using ast module (Abstract Syntax Tree)
def parse_choices(x):
    # If x is already a list, return it
    if isinstance(x, list):
        return x
    # If x is a string, parse it using ast.literal_eval
    return ast.literal_eval(x)

The ```choices``` column is a JSON string, so we parse it using the function above.

In [12]:
for df in [train_df, val_df, test_df]:
    df["choices_list"] = df["choices"].apply(parse_choices)

---
### Prompt

Keep `build_prompt_model2_imagecareful` which is a variation of Model 2's prompt.

In [13]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

In [14]:
def build_prompt(row):
    choices = row["choices_list"]

    choice_lines = "\n".join(
        [f"{i}: {choice}" for i, choice in enumerate(choices)]
    )

    prompt = "<image>\n"
    prompt += "Answer the science multiple-choice question.\n"
    prompt += "Look carefully at the image when it is relevant.\n"
    prompt += "Return only the number of the correct choice.\n\n"

    prompt += f"Grade: {clean_text(row.get('grade', ''))}\n"
    prompt += f"Subject: {clean_text(row.get('subject', ''))}\n"
    prompt += f"Topic: {clean_text(row.get('topic', ''))}\n\n"

    hint = clean_text(row.get("hint", ""))
    if hint:
        prompt += f"Hint:\n{hint}\n\n"

    prompt += f"Question:\n{clean_text(row['question'])}\n\n"
    prompt += f"Choices:\n{choice_lines}\n\n"
    prompt += "Answer:"

    return prompt

---
### Model Loading

In [15]:
# The model is not related to Models 1, 2, and 3.
processor = AutoProcessor.from_pretrained(MODEL_ID)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

for p in model.parameters():
    p.requires_grad = False

# This time, we will attach LoRA
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

# Make sure the number of trainable parameters is less than 5 million
assert trainable < 5_000_000

/home/devvingduo/miniforge3/envs/pixels-to-predictions/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Trainable params: 557,056
Total params: 508,039,360
Trainable %: 0.1096%


---
### Index-Only Scoring
Index-onluy scoring did better than hybrid index+answer scoring. This keeps the Model 2 idea, but uses safer answer-token masking.

In [16]:
@torch.no_grad()
def score_indices_for_row(row, prompt_builder, score_mode="sum"):
    image = Image.open(IMAGE_ROOT / row["image_path"]).convert("RGB")
    prompt = prompt_builder(row)

    num_choices = int(row["num_choices"])
    answer_texts = [f" {i}" for i in range(num_choices)]
    full_texts = [prompt + ans for ans in answer_texts]

    prompt_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=True,
    )

    prompt_len = prompt_inputs["input_ids"].shape[1]

    full_inputs = processor(
        text=full_texts,
        images=[image] * num_choices,
        return_tensors="pt",
        padding=True,
    )

    full_inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in full_inputs.items()
    }

    input_ids = full_inputs["input_ids"]
    attention_mask = full_inputs.get("attention_mask", torch.ones_like(input_ids))

    outputs = model(**full_inputs)
    logits = outputs.logits.float()

    shift_logits = logits[:, :-1, :]
    shift_input_ids = input_ids[:, 1:]
    shift_attention = attention_mask[:, 1:]

    log_probs = F.log_softmax(shift_logits, dim=-1)

    token_log_probs = log_probs.gather(
        dim=-1,
        index=shift_input_ids.unsqueeze(-1),
    ).squeeze(-1)

    answer_mask = torch.zeros_like(shift_input_ids, dtype=torch.bool)

    # Because logits[:, t] predicts token input_ids[:, t+1],
    # answer tokens begin around prompt_len - 1 in shifted space.
    start = max(prompt_len - 1, 0)
    answer_mask[:, start:] = True

    # Ignore padding.
    answer_mask = answer_mask & shift_attention.bool()

    scores = []

    for i in range(num_choices):
        vals = token_log_probs[i][answer_mask[i]]

        if vals.numel() == 0:
            scores.append(-1e9)
        elif score_mode == "sum":
            scores.append(vals.sum().item())
        elif score_mode == "mean":
            scores.append(vals.mean().item())
        else:
            raise ValueError("score_mode must be 'sum' or 'mean'")

    return scores

In [17]:
# Since we will train the model this time, we need to use the training version of the scoring function.
# Same idea as Model 4 scoring, but WITHOUT torch.no_grad(), so LoRA can learn from the choice scores.
def score_indices_for_row_train(row, prompt_builder, score_mode="sum"):
    prompt = prompt_builder(row)
    choices = row["choices_list"]
    num_choices = int(row["num_choices"])

    image_path = IMAGE_ROOT / row["image_path"]
    image = Image.open(image_path).convert("RGB")

    scores = []

    # For each choice, append the choice index to the prompt and compute the score
    for choice_idx in range(num_choices):
        answer_text = " " + str(choice_idx)
        full_text = prompt + answer_text

        inputs = processor(
            text=[full_text],
            images=[image],
            return_tensors="pt",
        )

        model_device = next(model.parameters()).device
        inputs = {
            k: v.to(model_device) if torch.is_tensor(v) else v
            for k, v in inputs.items()
        }

        input_ids = inputs["input_ids"]

        prompt_inputs = processor(
            text=[prompt],
            images=[image],
            return_tensors="pt",
        )

        prompt_len = prompt_inputs["input_ids"].shape[1]

        outputs = model(**inputs)
        logits = outputs.logits.float()

        # logits[:, t] predicts input_ids[:, t+1]
        log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
        target_ids = input_ids[:, 1:]

        token_log_probs = log_probs.gather(
            dim=2,
            index=target_ids.unsqueeze(-1),
        ).squeeze(-1)

        start = max(prompt_len - 1, 0)
        answer_token_log_probs = token_log_probs[:, start:]
        score = answer_token_log_probs.sum()

        scores.append(score)

    # Return the scores as a tensor
    return torch.stack(scores)

In [18]:
def predict_row(row, prompt_builder, score_mode="sum"):
    scores = score_indices_for_row(row, prompt_builder, score_mode=score_mode)
    pred = int(np.argmax(scores))
    return pred, scores

---
### Evaluation

Contrastive LoRA training scores every answer index for the same question, treats those scores as classification logits, and uses cross-entropy to push the correct answer’s score above the wrong choices. 

In [19]:
# Score all answer indices, then apply cross entropy to make the correct answer index score highest
def contrastive_loss_for_row(row, prompt_builder=build_prompt):
    scores = score_indices_for_row_train(
        row,
        prompt_builder=prompt_builder,
        score_mode="sum",
    )

    logits = scores.unsqueeze(0)

    target = torch.tensor(
        [int(row["answer"])],
        dtype=torch.long,
        device=logits.device,
    )

    loss = F.cross_entropy(logits, target)

    pred = int(torch.argmax(scores.detach()).item())

    return loss, pred, scores.detach().cpu().numpy()

In [20]:
def evaluate_model(df=None, max_rows=None, score_mode="sum"):
    if df is None:
        df = val_df

    if max_rows is not None:
        eval_df = df.sample(max_rows, random_state=SEED).reset_index(drop=True)
    else:
        eval_df = df.reset_index(drop=True)

    preds = []
    labels = []
    all_scores = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="evaluate"):
        pred, scores = predict_row(
            row,
            build_prompt,
            score_mode=score_mode,
        )
        preds.append(pred)
        labels.append(int(row["answer"]))
        all_scores.append(scores)

    preds = np.array(preds)
    labels = np.array(labels)
    acc = np.mean(preds == labels)

    print(f"build_prompt | {score_mode} | accuracy = {acc:.4f}")

    print("Prediction distribution:")
    display(pd.Series(preds).value_counts(normalize=True).sort_index())

    return {
        "score_mode": score_mode,
        "accuracy": acc,
        "preds": preds,
        "labels": labels,
        "scores": all_scores,
        "eval_df": eval_df,
    }

In [21]:
# baseline_full = evaluate_model(
#     val_df,
#     max_rows=None,
#     score_mode="sum",
# )

Before running full training for several epochs, run a small sample sanity-check.

In [22]:
# TINY_TRAIN_N = 64
# TINY_VAL_N = 200

# tiny_train_df = train_df.sample(
#     n=TINY_TRAIN_N,
#     random_state=42,
# ).reset_index(drop=True)

# optimizer = torch.optim.AdamW(
#     [p for p in model.parameters() if p.requires_grad],
#     lr=1e-5,
#     weight_decay=0.01,
# )

# grad_accum_steps = 8

# model.train()
# optimizer.zero_grad(set_to_none=True)

# losses = []
# preds = []
# labels = []

# for step, (_, row) in enumerate(tqdm(tiny_train_df.iterrows(), total=len(tiny_train_df), desc="tiny train")):
#     loss, pred, scores = contrastive_loss_for_row(row)

#     (loss / grad_accum_steps).backward()

#     losses.append(float(loss.detach().cpu()))
#     preds.append(pred)
#     labels.append(int(row["answer"]))

#     if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(tiny_train_df):
#         torch.nn.utils.clip_grad_norm_(
#             [p for p in model.parameters() if p.requires_grad],
#             max_norm=1.0,
#         )
#         optimizer.step()
#         optimizer.zero_grad(set_to_none=True)
#         torch.cuda.empty_cache()

#     if step < 5:
#         print()
#         print("target:", int(row["answer"]), "pred:", pred)
#         print("scores:", np.round(scores, 4))

# print("Mean tiny loss:", np.mean(losses))
# print("Tiny train acc:", np.mean(np.array(preds) == np.array(labels)))
# print("Tiny pred distribution:")
# display(pd.Series(preds).value_counts(normalize=True).sort_index())

In [23]:
# after_tiny_full = evaluate_model(
#     val_df,
#     max_rows=None,
#     score_mode="sum",
# )

# if after_tiny_full["accuracy"] >= baseline_full["accuracy"]:
#     print("Tiny training did not hurt.")
# else:
#     print("Tiny training hurt validation. Do not run full training.")

---
### Full Training

I'll leave the model to train overnight for several epochs. It will save after each epoch so I can review and choose the best version later.

In [24]:
# FULL_EPOCHS = 3
# LR = 1e-5
# GRAD_ACCUM_STEPS = 8

# MODEL4_TARGET_VAL = 0.5811068702290076

# BEST_DIR = Path("../runs/model5_contrastive_lora_best")
# LATEST_DIR = Path("../runs/model5_contrastive_lora_latest")
# BEST_DIR.mkdir(exist_ok=True, parents=True)
# LATEST_DIR.mkdir(exist_ok=True, parents=True)

# optimizer = torch.optim.AdamW(
#     [p for p in model.parameters() if p.requires_grad],
#     lr=LR,
#     weight_decay=0.01,
# )

# # Does NOT depend on baseline_full anymore
# best_model5_val_acc = -1.0
# history = []

# for epoch in range(1, FULL_EPOCHS + 1):
#     model.train()
#     optimizer.zero_grad(set_to_none=True)

#     train_order = train_df.sample(
#         frac=1,
#         random_state=42 + epoch,
#     ).reset_index(drop=True)

#     losses = []
#     preds = []
#     labels = []

#     for step, (_, row) in enumerate(
#         tqdm(train_order.iterrows(), total=len(train_order), desc=f"Model 5 epoch {epoch}")
#     ):
#         loss, pred, scores = contrastive_loss_for_row(row)

#         (loss / GRAD_ACCUM_STEPS).backward()

#         losses.append(float(loss.detach().cpu()))
#         preds.append(pred)
#         labels.append(int(row["answer"]))

#         if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_order):
#             torch.nn.utils.clip_grad_norm_(
#                 [p for p in model.parameters() if p.requires_grad],
#                 max_norm=1.0,
#             )
#             optimizer.step()
#             optimizer.zero_grad(set_to_none=True)
#             torch.cuda.empty_cache()

#         if (step + 1) % 50 == 0:
#             recent_loss = np.mean(losses[-50:])
#             recent_acc = np.mean(np.array(preds[-100:]) == np.array(labels[-100:]))
#             print(f"epoch {epoch}, step {step+1}, recent_loss={recent_loss:.4f}, recent_acc={recent_acc:.4f}")

#     train_acc = float(np.mean(np.array(preds) == np.array(labels)))
#     mean_loss = float(np.mean(losses))

#     print()
#     print(f"Epoch {epoch} train loss:", mean_loss)
#     print(f"Epoch {epoch} train acc:", train_acc)
#     print("Train pred distribution:")
#     display(pd.Series(preds).value_counts(normalize=True).sort_index())

#     model.eval()

#     val_result = evaluate_model(
#         val_df,
#         max_rows=None,
#         score_mode="sum",
#     )

#     val_acc = float(val_result["accuracy"])

#     # Save latest every epoch no matter what
#     model.save_pretrained(LATEST_DIR)
#     processor.save_pretrained(LATEST_DIR)

#     is_best = val_acc > best_model5_val_acc

#     row = {
#         "epoch": epoch,
#         "train_loss": mean_loss,
#         "train_acc": train_acc,
#         "val_acc": val_acc,
#         "saved_best": is_best,
#         "beats_model4_target": val_acc > MODEL4_TARGET_VAL,
#     }

#     history.append(row)
#     history_df = pd.DataFrame(history)
#     history_df.to_csv("../runs/model5_training_history.csv", index=False)

#     display(history_df)

#     if is_best:
#         best_model5_val_acc = val_acc
#         model.save_pretrained(BEST_DIR)
#         processor.save_pretrained(BEST_DIR)
#         print("Saved new BEST Model 5 adapter:", BEST_DIR)
#     else:
#         print("Did not beat previous Model 5 best. Latest checkpoint still saved.")

#     print("Current best Model 5 val:", best_model5_val_acc)
#     print("Model 4 target val:", MODEL4_TARGET_VAL)

# print("Training finished.")
# print("Best Model 5 val:", best_model5_val_acc)
# print("Model 4 target val:", MODEL4_TARGET_VAL)

# if best_model5_val_acc > MODEL4_TARGET_VAL:
#     print("Model 5 beat Model 4 target. Use best adapter for submission.")
# else:
#     print("Model 5 did not beat Model 4 target. Keep Model 4 submission.")

I got disconnected from SSH overnight, but thankfully we saved some weights. Continue training:

In [25]:
MODEL4_TARGET_VAL = 0.5811068702290076

BEST_DIR = Path("../runs/model5_contrastive_lora_best")
LATEST_DIR = Path("../runs/model5_contrastive_lora_latest")
HISTORY_PATH = Path("../runs/model5_training_history.csv")

# Load training history
history_df = pd.read_csv(HISTORY_PATH)
display(history_df)

start_epoch = int(history_df["epoch"].max()) + 1
best_model5_val_acc = float(history_df["val_acc"].max())
history = history_df.to_dict("records")

print("Resuming from epoch:", start_epoch)
print("Current best val:", best_model5_val_acc)

,epoch,train_loss,train_acc,val_acc,saved_best,beats_model4_target
0,1,0.972028,0.633001,0.609733,True,True
1,2,0.755157,0.671920,0.642176,True,True


Resuming from epoch: 3
Current best val: 0.642175572519084


In [27]:
model = PeftModel.from_pretrained(
    model,
    LATEST_DIR,
    is_trainable=True,
)

model.train()

/home/devvingduo/miniforge3/envs/pixels-to-predictions/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/devvingduo/miniforge3/envs/pixels-to-predictions/lib/python3.10/site-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.vision_model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.vision_model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.vision_model.encoder.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.vision_model.encoder.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.vision_model.encoder.layers.1.self_a

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): Idefics3ForConditionalGeneration(
          (model): Idefics3Model(
            (vision_model): Idefics3VisionTransformer(
              (embeddings): Idefics3VisionEmbeddings(
                (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
                (position_embedding): Embedding(1024, 768)
              )
              (encoder): Idefics3Encoder(
                (layers): ModuleList(
                  (0-11): 12 x Idefics3EncoderLayer(
                    (self_attn): Idefics3VisionAttention(
                      (k_proj): Linear(in_features=768, out_features=768, bias=True)
                      (v_proj): lora.Linear(
                        (base_layer): Linear(in_features=768, out_features=768, bias=True)
                        (lora_dropout): ModuleDict(
                          (default): Dropout(p=0

In [28]:
FULL_EPOCHS = 3
LR = 1e-5
GRAD_ACCUM_STEPS = 8

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

for epoch in range(start_epoch, FULL_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    train_order = train_df.sample(
        frac=1,
        random_state=42 + epoch,
    ).reset_index(drop=True)

    losses = []
    preds = []
    labels = []

    for step, (_, row) in enumerate(
        tqdm(train_order.iterrows(), total=len(train_order), desc=f"Model 5 epoch {epoch}")
    ):
        loss, pred, scores = contrastive_loss_for_row(row)

        (loss / GRAD_ACCUM_STEPS).backward()

        losses.append(float(loss.detach().cpu()))
        preds.append(pred)
        labels.append(int(row["answer"]))

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_order):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()

        if (step + 1) % 50 == 0:
            recent_loss = np.mean(losses[-50:])
            recent_acc = np.mean(np.array(preds[-100:]) == np.array(labels[-100:]))
            print(f"epoch {epoch}, step {step+1}, recent_loss={recent_loss:.4f}, recent_acc={recent_acc:.4f}")

        # Optional safety checkpoint every 500 steps
        if (step + 1) % 500 == 0:
            STEP_DIR = Path(f"../runs/model5_step_checkpoint_epoch{epoch}_step{step+1}")
            STEP_DIR.mkdir(exist_ok=True, parents=True)
            model.save_pretrained(STEP_DIR)
            processor.save_pretrained(STEP_DIR)
            print("Saved step checkpoint:", STEP_DIR)

    train_acc = float(np.mean(np.array(preds) == np.array(labels)))
    mean_loss = float(np.mean(losses))

    print()
    print(f"Epoch {epoch} train loss:", mean_loss)
    print(f"Epoch {epoch} train acc:", train_acc)
    print("Train pred distribution:")
    display(pd.Series(preds).value_counts(normalize=True).sort_index())

    model.eval()

    val_result = evaluate_model(
        val_df,
        max_rows=None,
        score_mode="sum",
    )

    val_acc = float(val_result["accuracy"])

    # Save latest every epoch
    model.save_pretrained(LATEST_DIR)
    processor.save_pretrained(LATEST_DIR)

    is_best = val_acc > best_model5_val_acc

    row = {
        "epoch": epoch,
        "train_loss": mean_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "saved_best": is_best,
        "beats_model4_target": val_acc > MODEL4_TARGET_VAL,
    }

    history.append(row)
    history_df = pd.DataFrame(history)
    history_df.to_csv(HISTORY_PATH, index=False)

    display(history_df)

    if is_best:
        best_model5_val_acc = val_acc
        model.save_pretrained(BEST_DIR)
        processor.save_pretrained(BEST_DIR)
        print("Saved new BEST Model 5 adapter:", BEST_DIR)
    else:
        print("Did not beat previous Model 5 best. Latest checkpoint still saved.")

    print("Current best Model 5 val:", best_model5_val_acc)
    print("Model 4 target val:", MODEL4_TARGET_VAL)

print("Training finished.")
print("Best Model 5 val:", best_model5_val_acc)

Model 5 epoch 3:   0%|          | 0/3109 [00:00<?, ?it/s]

epoch 3, step 50, recent_loss=1.4049, recent_acc=0.6000
epoch 3, step 100, recent_loss=1.0589, recent_acc=0.6200
epoch 3, step 150, recent_loss=1.4016, recent_acc=0.6400
epoch 3, step 200, recent_loss=1.2055, recent_acc=0.6100
epoch 3, step 250, recent_loss=1.1229, recent_acc=0.6400
epoch 3, step 300, recent_loss=1.1397, recent_acc=0.6500
epoch 3, step 350, recent_loss=1.0590, recent_acc=0.6300
epoch 3, step 400, recent_loss=1.0227, recent_acc=0.6200
epoch 3, step 450, recent_loss=1.0881, recent_acc=0.5900
epoch 3, step 500, recent_loss=1.1902, recent_acc=0.5900
Saved step checkpoint: ../runs/model5_step_checkpoint_epoch3_step500
epoch 3, step 550, recent_loss=0.9722, recent_acc=0.5900
epoch 3, step 600, recent_loss=1.0278, recent_acc=0.6400
epoch 3, step 650, recent_loss=0.8811, recent_acc=0.6900
epoch 3, step 700, recent_loss=0.9855, recent_acc=0.6700
epoch 3, step 750, recent_loss=0.9521, recent_acc=0.6400
epoch 3, step 800, recent_loss=1.1730, recent_acc=0.6500
epoch 3, step 850, r

0    0.357028
1    0.281441
2    0.310068
3    0.051463
Name: proportion, dtype: float64

evaluate:   0%|          | 0/1048 [00:00<?, ?it/s]

build_prompt | sum | accuracy = 0.6116
Prediction distribution:


0    0.357824
1    0.320611
2    0.285305
3    0.036260
Name: proportion, dtype: float64

,epoch,train_loss,train_acc,val_acc,saved_best,beats_model4_target
0,1,0.972028,0.633001,0.609733,True,True
1,2,0.755157,0.671920,0.642176,True,True
2,3,0.969861,0.637182,0.611641,False,True


Did not beat previous Model 5 best. Latest checkpoint still saved.
Current best Model 5 val: 0.642175572519084
Model 4 target val: 0.5811068702290076
Training finished.
Best Model 5 val: 0.642175572519084


In [31]:
BEST_DIR = Path("../runs/model5_contrastive_lora_best")

# Clear current epoch-3/latest model from memory
try:
    del model
except:
    pass

torch.cuda.empty_cache()

# Reload fresh base model
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

# Load BEST LoRA adapter, which should be epoch 2
model = PeftModel.from_pretrained(
    base_model,
    BEST_DIR,
    is_trainable=False,
)

model.eval()

print("Loaded best Model 5 adapter from:", BEST_DIR)

/home/devvingduo/miniforge3/envs/pixels-to-predictions/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loaded best Model 5 adapter from: ../runs/model5_contrastive_lora_best


In [32]:
history_df = pd.read_csv("../runs/model5_training_history.csv")
display(history_df)

best_epoch = history_df.loc[history_df["val_acc"].idxmax()]
print("Best epoch:")
display(best_epoch)

,epoch,train_loss,train_acc,val_acc,saved_best,beats_model4_target
0,1,0.972028,0.633001,0.609733,True,True
1,2,0.755157,0.671920,0.642176,True,True
2,3,0.969861,0.637182,0.611641,False,True


Best epoch:


epoch                         2
train_loss             0.755157
train_acc               0.67192
val_acc                0.642176
saved_best                 True
beats_model4_target        True
Name: 1, dtype: object

---
### Submission

In [33]:
def predict_test(score_mode="sum"):
    preds = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="test"):
        pred, scores = predict_row(
            row,
            build_prompt,
            score_mode=score_mode,
        )
        preds.append(pred)

    submission = pd.DataFrame({
        "id": test_df["id"].values,
        "answer": np.array(preds, dtype=int),
    })

    return submission

In [34]:
submission = predict_test(score_mode="sum")

display(submission.head())
display(submission["answer"].value_counts(normalize=True).sort_index())

test:   0%|          | 0/1008 [00:00<?, ?it/s]

,id,answer
0,test_01750,1
1,test_00128,0
2,test_02891,2
3,test_02425,0
4,test_00930,0


answer
0    0.423611
1    0.310516
2    0.209325
3    0.053571
4    0.002976
Name: proportion, dtype: float64

In [ ]:
def validate_submission(submission, test_df):
    assert list(submission.columns) == ["id", "answer"], "Submission must have columns: id, answer"
    assert len(submission) == len(test_df), "Submission has wrong number of rows"
    assert set(submission["id"]) == set(test_df["id"]), "Submission IDs do not match test IDs"
    assert submission["answer"].isna().sum() == 0, "Submission has missing answers"

    submission["answer"] = submission["answer"].astype(int)

    check_df = submission.merge(
        test_df[["id", "num_choices"]],
        on="id",
        how="left",
    )

    bad = check_df[
        (check_df["answer"] < 0) |
        (check_df["answer"] >= check_df["num_choices"])
    ]

    assert len(bad) == 0, f"Invalid answer indices found:\n{bad.head()}"

    print("Submission validation passed.")

submission = submission[["id", "answer"]].copy()
submission["answer"] = submission["answer"].astype(int)

validate_submission(submission, test_df)